In [4]:
# severity_segregate.py
# Scores ulcer images and splits into risk folders based on ACTUAL score distribution
# Uses percentile-based thresholds so every class always gets images

import cv2
import numpy as np
import shutil
import csv
from pathlib import Path
from datetime import datetime

# PATHS
ULCER_DIR    = Path("D:/Dia Foot/DFU/Diabetic Foot Ulcer")
NORMAL_DIR   = Path("D:/Dia Foot/DFU/Normal")
ROOT         = Path("D:/Dia Foot/DFU")
HIGH_OUT     = ROOT / "high_risk"
MODERATE_OUT = ROOT / "moderate"
LOW_OUT      = ROOT / "low_risk"
NORMAL_OUT   = ROOT / "normal"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# PERCENTILE SPLIT
# Top 25% of scores    -> high_risk
# Next 35% of scores   -> moderate
# Bottom 40% of scores -> low_risk
HIGH_PERCENTILE     = 75   # top 25%
MODERATE_PERCENTILE = 40   # next 35%

# HSV THRESHOLDS
DARK_VAL_MAX   = 80    # raised from 60 → catches more dark tissue
RED_HUE_HIGH_1 = 15    # raised from 10 → catches more red hues
RED_HUE_LOW_2  = 160   # lowered from 165 → catches more red hues
RED_SAT_MIN    = 30    # lowered from 50 → catches more red pixels


def compute_severity_score(img_path):
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        return -1.0
    bgr  = cv2.resize(bgr, (128, 128))
    hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    total   = h.size

    # Dark / necrotic tissue
    dark_ratio   = (v < DARK_VAL_MAX).sum() / total

    # Red / inflamed tissue
    red_mask     = ((h <= RED_HUE_HIGH_1) | (h >= RED_HUE_LOW_2)) & (s >= RED_SAT_MIN)
    red_ratio    = red_mask.sum() / total

    # Yellow / slough tissue (hue 15-40)
    yellow_mask  = ((h >= 15) & (h <= 40)) & (s >= 40) & (v >= 60)
    yellow_ratio = yellow_mask.sum() / total

    # Texture roughness
    gray         = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    lap          = cv2.Laplacian(gray, cv2.CV_64F)
    texture_norm = min(float(np.var(lap)) / 2000.0, 1.0)

    # Combined score (added yellow signal)
    score = (dark_ratio * 40) + (red_ratio * 30) + (yellow_ratio * 15) + (texture_norm * 15)
    return round(float(score), 4)


print("=" * 55)
print("  DFU Severity Segregator  (percentile-based)")
print("=" * 55)
print(f"  Ulcer  exists : {ULCER_DIR.exists()}")
print(f"  Normal exists : {NORMAL_DIR.exists()}")
print()

ulcer_files = sorted([
    f for f in ULCER_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in IMG_EXTS
])
normal_files = sorted([
    f for f in NORMAL_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in IMG_EXTS
])
print(f"  Ulcer images  : {len(ulcer_files)}")
print(f"  Normal images : {len(normal_files)}")
print()

for folder in [HIGH_OUT, MODERATE_OUT, LOW_OUT, NORMAL_OUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Score every ulcer image
print("Scoring ulcer images ...")
scored = []
errors = 0
for i, img_path in enumerate(ulcer_files, 1):
    score = compute_severity_score(img_path)
    if score < 0:
        errors += 1
        continue
    scored.append({"path": img_path, "score": score})
    if i % 200 == 0:
        print(f"  ... {i} / {len(ulcer_files)} scored")

print(f"  [DONE] {len(scored)} scored  |  {errors} errors\n")

# Sort by score descending
scored.sort(key=lambda x: x["score"], reverse=True)

# Show actual score range
all_scores = [item["score"] for item in scored]
print(f"  Score range : min={min(all_scores):.2f}  max={max(all_scores):.2f}  mean={np.mean(all_scores):.2f}")

# Calculate percentile thresholds from actual data
high_threshold = float(np.percentile(all_scores, HIGH_PERCENTILE))
mod_threshold  = float(np.percentile(all_scores, MODERATE_PERCENTILE))
print(f"  Auto thresholds:")
print(f"    high_risk  : score >= {high_threshold:.2f}  (top 25%)")
print(f"    moderate   : score >= {mod_threshold:.2f}  (next 35%)")
print(f"    low_risk   : score <  {mod_threshold:.2f}  (bottom 40%)")
print()

# Assign labels using percentile thresholds
for item in scored:
    if item["score"] >= high_threshold:
        item["label"] = "high_risk"
    elif item["score"] >= mod_threshold:
        item["label"] = "moderate"
    else:
        item["label"] = "low_risk"

counts = {"high_risk": 0, "moderate": 0, "low_risk": 0}
for item in scored:
    counts[item["label"]] += 1

total = len(scored)
print(f"  high_risk  : {counts['high_risk']:>4} images  ({counts['high_risk']/total*100:.1f}%)")
print(f"  moderate   : {counts['moderate']:>4} images  ({counts['moderate']/total*100:.1f}%)")
print(f"  low_risk   : {counts['low_risk']:>4} images  ({counts['low_risk']/total*100:.1f}%)")
print()

# DOCTOR ALERT
high_risk_images = [item for item in scored if item["label"] == "high_risk"]
alert_lines = []
alert_lines.append("=" * 55)
alert_lines.append("  DOCTOR ALERT REPORT")
alert_lines.append(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
alert_lines.append("=" * 55)

if counts["high_risk"] > 0:
    print("!" * 55)
    print("  *** DOCTOR ALERT ***")
    print(f"  {counts['high_risk']} HIGH RISK patient image(s) detected!")
    print("  IMMEDIATE MEDICAL ATTENTION REQUIRED")
    print("!" * 55)
    print()
    print("  Clinical signs detected:")
    print("    - Necrotic / dead tissue (dark areas)")
    print("    - Deep wound bed inflammation (red areas)")
    print("    - High wound complexity")
    print()
    print(f"  Top 10 most severe images:")
    alert_lines.append("")
    alert_lines.append(f"  URGENT: {counts['high_risk']} HIGH RISK cases detected!")
    alert_lines.append("")
    alert_lines.append("  Recommended Actions:")
    alert_lines.append("    1. Immediate wound debridement")
    alert_lines.append("    2. Infection control assessment")
    alert_lines.append("    3. Vascular evaluation")
    alert_lines.append("    4. Consider hospitalisation")
    alert_lines.append("")
    alert_lines.append("  High Risk Image List (sorted by severity):")
    for rank, item in enumerate(high_risk_images[:10], 1):
        line = f"    #{rank:>2}  Score: {item['score']:>5.2f}  File: {item['path'].name}"
        print(line)
        alert_lines.append(line)
    if len(high_risk_images) > 10:
        print(f"    ... and {len(high_risk_images)-10} more in high_risk/ folder")

elif counts["moderate"] > 0:
    print("-" * 55)
    print(f"  [MODERATE] {counts['moderate']} case(s) need monitoring.")
    print("  Recommend follow-up within 1-2 weeks.")
    print("-" * 55)
    alert_lines.append("  STATUS: MODERATE - monitoring required")
    alert_lines.append(f"  {counts['moderate']} cases need follow-up within 1-2 weeks")

alert_lines.append("")
alert_lines.append(f"  high_risk : {counts['high_risk']} images")
alert_lines.append(f"  moderate  : {counts['moderate']} images")
alert_lines.append(f"  low_risk  : {counts['low_risk']} images")
alert_lines.append(f"  normal    : {len(normal_files)} images")

alert_path = ROOT / "alert_report.txt"
with open(alert_path, "w", encoding="utf-8") as f:
    f.write("\n".join(alert_lines))
print(f"\n  Alert report saved -> {alert_path}\n")

# Copy ulcer images
report_rows    = []
label_counters = {"high_risk": 0, "moderate": 0, "low_risk": 0}
folder_map     = {"high_risk": HIGH_OUT, "moderate": MODERATE_OUT, "low_risk": LOW_OUT}

print("Copying ulcer images ...")
for item in scored:
    label  = item["label"]
    src    = item["path"]
    label_counters[label] += 1
    new_name = f"{label}_{label_counters[label]:04d}{src.suffix.lower()}"
    dest     = folder_map[label] / new_name
    shutil.copy2(str(src), str(dest))
    report_rows.append({"original": src.name, "new_name": new_name,
                        "label": label, "score": item["score"]})
print(f"  [DONE]\n")

# Copy normal images
print("Copying normal images ...")
skipped = 0
for idx, src in enumerate(normal_files, start=1):
    try:
        new_name = f"normal_{idx:04d}{src.suffix.lower()}"
        dest     = NORMAL_OUT / new_name
        shutil.copy2(str(src), str(dest))
        report_rows.append({"original": src.name, "new_name": new_name,
                            "label": "normal", "score": 0.0})
    except PermissionError:
        print(f"  [SKIP] locked: {src.name}")
        skipped += 1
print(f"  [DONE] {len(normal_files)-skipped} copied, {skipped} skipped\n")

# Save CSV
csv_path = ROOT / "severity_scores.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["original","new_name","label","score"])
    writer.writeheader()
    writer.writerows(report_rows)

print("=" * 55)
print("  FINAL SUMMARY")
print("=" * 55)
print(f"  high_risk/  ->  {counts['high_risk']:>4} images  *** URGENT ***")
print(f"  moderate/   ->  {counts['moderate']:>4} images")
print(f"  low_risk/   ->  {counts['low_risk']:>4} images")
print(f"  normal/     ->  {len(normal_files):>4} images")
print(f"  severity_scores.csv  saved")
print(f"  alert_report.txt     saved")

  DFU Severity Segregator  (percentile-based)
  Ulcer  exists : True
  Normal exists : True

  Ulcer images  : 2128
  Normal images : 543

Scoring ulcer images ...
  ... 200 / 2128 scored
  ... 400 / 2128 scored
  ... 600 / 2128 scored
  ... 800 / 2128 scored
  ... 1000 / 2128 scored
  ... 1200 / 2128 scored
  ... 1400 / 2128 scored
  ... 1600 / 2128 scored
  ... 1800 / 2128 scored
  ... 2000 / 2128 scored
  [DONE] 2128 scored  |  0 errors

  Score range : min=8.53  max=59.37  mean=33.28
  Auto thresholds:
    high_risk  : score >= 38.89  (top 25%)
    moderate   : score >= 31.42  (next 35%)
    low_risk   : score <  31.42  (bottom 40%)

  high_risk  :  533 images  (25.0%)
  moderate   :  744 images  (35.0%)
  low_risk   :  851 images  (40.0%)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  *** DOCTOR ALERT ***
  533 HIGH RISK patient image(s) detected!
  IMMEDIATE MEDICAL ATTENTION REQUIRED
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

  Clinical signs detected:
